# Домашняя работа 2. Линейная регрессия: синтетические и реальные данные

**Курс:** Машинное обучение и анализ данных

**Что сделано**

| Часть | Содержание |
|---|---|
| 0 | Импорты, настройки, вспомогательные функции |
| I (3.1–3.4) | Синтетические данные: генерация 3 зависимостей, полиномиальная регрессия, подбор степени, регуляризация (Ridge / Lasso), артефакты модели, выводы |
| II (4.1–4.5) | Реальные данные **California Housing**: EDA, предобработка, baseline, улучшение качества (полиномы, категориальные признаки, `TransformedTargetRegressor`, Ridge / Lasso / SGD), анализ ошибок, выводы |

**Принципы, которых я придерживаюсь**
- любое «обучаемое» преобразование (клиппинг выбросов, масштабирование, KMeans, one-hot) живёт внутри `Pipeline` и обучается **только на train** → нет утечки данных;
- модель выбирается по **кросс-валидации на train**, а тестовая выборка используется для финальной оценки;
- каждая картинка подписана и сопровождается комментарием, лишних графиков нет.

## 0. Импорты и настройки

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, PredictionErrorDisplay
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, PowerTransformer, OneHotEncoder

# Lasso / SGD на полиномах иногда предупреждают о сходимости — это ожидаемо и не влияет на выводы
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 42
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 12, "axes.labelsize": 10, "legend.fontsize": 9,
})

In [ ]:
def rmse(y_true, y_pred):
    # корень из MSE (в тех же единицах, что и целевая переменная)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics(y_true, y_pred):
    # три метрики, которые используем во всей работе
    return {"RMSE": rmse(y_true, y_pred),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred))}

# Часть I. Синтетические данные (3.1–3.4)

## 3.1 Генерация «интересных» зависимостей

Я взял три зависимости, **не повторяющие** примеры из демонстрационного блокнота. Каждая ломает линейную модель по-своему:

1. **Затухающие колебания с трендом**  $y(x)=e^{-x/4}\sin 2x + 0.1x$, $x\in[0,6]$ — основная функция, на ней разбираем всё подробно.
2. **Функция Рунге**  $y(x)=\dfrac{1}{1+25x^2}$, $x\in[-1,1]$ — классический пример, на котором полиномы «болтаются» у краёв.
3. **Модуль с высокочастотной «рябью»**  $y(x)=|x|+0.3\sin 8x$, $x\in[-1,1]$ — излом в нуле, гладкие полиномы его плохо аппроксимируют.

К каждой функции добавляется гауссов шум, а выборка делится на **обучающую (70 %)** и **валидационную (30 %)**. Выборки специально небольшие (70–100 точек): на малых данных эффекты переобучения и регуляризации видны особенно ярко.

In [ ]:
FUNCS = {
    "damped": dict(f=lambda x: np.exp(-x / 4) * np.sin(2 * x) + 0.1 * x,
                   lo=0.0, hi=6.0, noise=0.15, n=70, title=r"$y=e^{-x/4}\sin 2x+0.1x$"),
    "runge":  dict(f=lambda x: 1 / (1 + 25 * x ** 2),
                   lo=-1.0, hi=1.0, noise=0.04, n=100, title=r"$y=1/(1+25x^2)$"),
    "kink":   dict(f=lambda x: np.abs(x) + 0.3 * np.sin(8 * x),
                   lo=-1.0, hi=1.0, noise=0.05, n=100, title=r"$y=|x|+0.3\sin 8x$"),
}

def make_data(name, seed=SEED):
    # генерируем выборку по закону + шум и делим на train / val
    cfg = FUNCS[name]
    n = cfg["n"]
    rng = np.random.default_rng(seed)
    x = rng.uniform(cfg["lo"], cfg["hi"], n)
    y = cfg["f"](x) + rng.normal(0, cfg["noise"], n)
    return train_test_split(x.reshape(-1, 1), y, test_size=0.3, random_state=seed)

def poly_model(degree, regressor):
    # полиномиальные признаки -> стандартизация -> линейная модель
    return Pipeline([
        ("poly", PolynomialFeatures(degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", regressor),
    ])

X_tr, X_val, y_tr, y_val = make_data("damped")
cfg = FUNCS["damped"]
x_grid = np.linspace(cfg["lo"], cfg["hi"], 600).reshape(-1, 1)   # плотная сетка для гладких графиков

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_grid, cfg["f"](x_grid), "k-", lw=2, label="истинная зависимость")
ax.scatter(X_tr, y_tr, s=14, alpha=.7, label=f"train (n={len(y_tr)})")
ax.scatter(X_val, y_val, s=14, alpha=.7, label=f"val (n={len(y_val)})")
ax.set(title="Основная синтетическая выборка: " + cfg["title"], xlabel="x", ylabel="y")
ax.legend(); plt.show()

*Комментарий.* В данных видно ~2 периода затухающей синусоиды на фоне слабого роста. Прямая линия тут принципиально не справится — нужны нелинейные признаки.

## 3.2 Линейная регрессия и полиномы

### 3.2.1 Подбор степени полинома

Для степеней 1…20 обучаем `LinearRegression` на `PolynomialFeatures` и смотрим RMSE на train и val. Горизонтальная линия — уровень шума $\sigma$: ниже него честная валидационная ошибка опуститься не может (это неустранимая ошибка).

In [ ]:
degrees = np.arange(1, 21)
rows = []
for d in degrees:
    m = poly_model(d, LinearRegression()).fit(X_tr, y_tr)
    rows.append({"degree": d,
                 "train_RMSE": rmse(y_tr, m.predict(X_tr)),
                 "val_RMSE": rmse(y_val, m.predict(X_val))})
deg_df = pd.DataFrame(rows)
best_deg = int(deg_df.loc[deg_df["val_RMSE"].idxmin(), "degree"])
OVERFIT_DEG = 20

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# (а) кривые ошибок по степени
ax = axes[0]
ax.plot(deg_df["degree"], deg_df["train_RMSE"], "o-", label="train RMSE")
ax.plot(deg_df["degree"], deg_df["val_RMSE"], "s-", label="val RMSE")
ax.axhline(cfg["noise"], color="gray", ls="--", label=f"уровень шума σ={cfg['noise']}")
ax.axvline(best_deg, color="green", ls=":", label=f"лучшая степень = {best_deg}")
ax.set_yscale("log"); ax.set_xticks(degrees[::2])
ax.set(title="Ошибка в зависимости от степени полинома", xlabel="степень", ylabel="RMSE (лог. шкала)")
ax.legend()

# (б) как выглядят подгонки: недообучение / оптимум / переобучение
ax = axes[1]
ax.scatter(X_tr, y_tr, s=10, alpha=.4, color="gray", label="train")
ax.plot(x_grid, cfg["f"](x_grid), "k-", lw=2, label="истина")
for d, c in [(1, "tab:blue"), (best_deg, "tab:green"), (OVERFIT_DEG, "tab:red")]:
    m = poly_model(d, LinearRegression()).fit(X_tr, y_tr)
    ax.plot(x_grid, m.predict(x_grid), color=c, lw=1.8, label=f"степень {d}")
ax.set_ylim(-1.5, 1.5)
ax.set(title="Недообучение — оптимум — переобучение", xlabel="x", ylabel="y")
ax.legend(); plt.show()

display(deg_df.round(4).set_index("degree").T.iloc[:, [0, 1, 2, 3, 4, 6, 8, 10, 12, 14, 16, 19]])

*Комментарий.* Train-ошибка монотонно падает с ростом степени, а val-ошибка имеет U-образную форму: слева — **недообучение** (смещение, модель слишком проста), справа — **переобучение** (дисперсия, полином «ловит» шум и раскачивается на краях). Оптимум достигается примерно там, где val-RMSE приближается к уровню шума.

### 3.2.2 Регуляризация: Ridge и Lasso

Зафиксируем **заведомо избыточную** степень (15) и посмотрим, может ли регуляризация вытащить модель из переобучения. Подбираем $\alpha$ по val-RMSE.

- **Ridge** (L2) уменьшает все веса, но не зануляет.
- **Lasso** (L1) зануляет часть весов → отбор признаков.

In [ ]:
REG_DEG = 15
ridge_alphas = np.logspace(-6, 2, 17)
lasso_alphas = np.logspace(-5, 0, 16)

def sweep(make_reg, alphas):
    out = []
    for a in alphas:
        m = poly_model(REG_DEG, make_reg(a)).fit(X_tr, y_tr)
        out.append((rmse(y_tr, m.predict(X_tr)), rmse(y_val, m.predict(X_val))))
    return np.array(out)

ridge_curve = sweep(lambda a: Ridge(alpha=a), ridge_alphas)
lasso_curve = sweep(lambda a: Lasso(alpha=a, max_iter=100000, tol=1e-4), lasso_alphas)

best_ridge_a = ridge_alphas[ridge_curve[:, 1].argmin()]
best_lasso_a = lasso_alphas[lasso_curve[:, 1].argmin()]

# итоговые модели для сравнения
models_syn = {
    "LinReg, степень 1 (недообучение)": poly_model(1, LinearRegression()),
    f"LinReg, степень {best_deg} (лучшая по val)": poly_model(best_deg, LinearRegression()),
    f"LinReg, степень {REG_DEG} (без регуляризации)": poly_model(REG_DEG, LinearRegression()),
    f"Ridge α={best_ridge_a:.1e}, степень {REG_DEG}": poly_model(REG_DEG, Ridge(alpha=best_ridge_a)),
    f"Lasso α={best_lasso_a:.1e}, степень {REG_DEG}": poly_model(REG_DEG, Lasso(alpha=best_lasso_a, max_iter=100000, tol=1e-4)),
}
for m in models_syn.values():
    m.fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ax = axes[0]
ax.plot(ridge_alphas, ridge_curve[:, 1], "o-", label="Ridge: val")
ax.plot(ridge_alphas, ridge_curve[:, 0], "o--", alpha=.5, label="Ridge: train")
ax.plot(lasso_alphas, lasso_curve[:, 1], "s-", label="Lasso: val")
ax.plot(lasso_alphas, lasso_curve[:, 0], "s--", alpha=.5, label="Lasso: train")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set(title=f"Влияние силы регуляризации (степень {REG_DEG})", xlabel="α", ylabel="RMSE")
ax.legend()

ax = axes[1]
ax.scatter(X_tr, y_tr, s=10, alpha=.35, color="gray")
ax.plot(x_grid, cfg["f"](x_grid), "k-", lw=2, label="истина")
for (name, m), c, lab in zip(list(models_syn.items())[2:], ["tab:red", "tab:green", "tab:purple"],
                             ["LinReg (α=0)", f"Ridge (α={best_ridge_a:.0e})", f"Lasso (α={best_lasso_a:.0e})"]):
    ax.plot(x_grid, m.predict(x_grid), color=c, lw=1.8, label=lab)
ax.set_ylim(-1.5, 1.5)
ax.set(title=f"Подгонка при степени {REG_DEG}", xlabel="x", ylabel="y"); ax.legend()

ax = axes[2]
w = {k: m.named_steps["model"].coef_ for k, m in list(models_syn.items())[2:]}
idx = np.arange(1, REG_DEG + 1); width = 0.27
for i, (k, c, lab) in enumerate(zip(w, ["tab:red", "tab:green", "tab:purple"], ["LinReg", "Ridge", "Lasso"])):
    ax.bar(idx + (i - 1) * width, w[k], width, color=c, label=lab)
ax.set_yscale("symlog", linthresh=1e-2)
ax.set(title="Веса при x^k (стандартизованные признаки, symlog)", xlabel="k", ylabel="вес")
ax.legend(); plt.show()

*Комментарий.* Без регуляризации веса при высоких степенях велики и разнознаковы (они компенсируют друг друга) — типичный признак мультиколлинеарности и переобучения. Ridge сжимает веса, Lasso дополнительно зануляет часть из них; правый график в симлог-шкале показывает разницу в порядках величины. Точные значения см. в таблице ниже.

### 3.2.3 Остальные функции

Та же схема на двух других зависимостях: подбираем степень по val и сравниваем с Ridge на степени 20 (α по val).

In [ ]:
def study(name):
    Xa, Xb, ya, yb = make_data(name)
    c = FUNCS[name]
    ds = np.arange(1, 21)
    vals = [rmse(yb, poly_model(d, LinearRegression()).fit(Xa, ya).predict(Xb)) for d in ds]
    d_best = int(ds[int(np.argmin(vals))])
    alphas = np.logspace(-6, 2, 17)
    rv = [rmse(yb, poly_model(20, Ridge(alpha=a)).fit(Xa, ya).predict(Xb)) for a in alphas]
    a_best = alphas[int(np.argmin(rv))]
    return dict(name=name, cfg=c, Xa=Xa, ya=ya, Xb=Xb, yb=yb, d_best=d_best, lin_val=min(vals),
                a_best=a_best, ridge_val=min(rv))

other = [study("runge"), study("kink")]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, s in zip(axes, other):
    c = s["cfg"]; g = np.linspace(c["lo"], c["hi"], 600).reshape(-1, 1)
    ax.scatter(s["Xa"], s["ya"], s=10, alpha=.35, color="gray", label="train")
    ax.plot(g, c["f"](g), "k-", lw=2, label="истина")
    ax.plot(g, poly_model(s["d_best"], LinearRegression()).fit(s["Xa"], s["ya"]).predict(g), color="tab:green",
            label=f"LinReg, степень {s['d_best']}")
    ax.plot(g, poly_model(20, Ridge(alpha=s["a_best"])).fit(s["Xa"], s["ya"]).predict(g), color="tab:purple", ls="--",
            label=f"Ridge, степень 20, α={s['a_best']:.0e}")
    ax.set(title=c["title"], xlabel="x", ylabel="y"); ax.legend()
plt.show()

pd.DataFrame([{"функция": s["name"], "σ шума": s["cfg"]["noise"], "лучшая степень": s["d_best"],
               "val RMSE (LinReg)": round(s["lin_val"], 4), "лучшее α (Ridge, deg 20)": f"{s['a_best']:.0e}",
               "val RMSE (Ridge)": round(s["ridge_val"], 4)} for s in other])

*Комментарий.* Полиномы высокой степени «звенят» у краёв отрезка (феномен Рунге) — это видно на правых границах обоих графиков. Важный нюанс: α подбирается по val-RMSE на **случайных** точках, которые редко попадают в самый край, поэтому выбранная α может быть слишком слабой и краевые артефакты остаются. Это ограничение метода валидации, а не регуляризации; надёжнее брать более сильное α или использовать базис, не склонный к колебаниям (сплайны). На функции с изломом любой гладкий полином остаётся лишь приближением.

## 3.3 Артефакты полученной модели

Для основной функции (`damped`) покажем: **веса**, **метрики**, **визуализацию**, **анализ остатков**, а также проверим, что веса sklearn совпадают с аналитическим решением МНК.

In [ ]:
# --- метрики всех сравниваемых моделей
rows = []
for name, m in models_syn.items():
    mt, mv = metrics(y_tr, m.predict(X_tr)), metrics(y_val, m.predict(X_val))
    rows.append({"модель": name, "train RMSE": mt["RMSE"], "val RMSE": mv["RMSE"],
                 "train R²": mt["R2"], "val R²": mv["R2"], "val MAE": mv["MAE"]})
syn_metrics = pd.DataFrame(rows).set_index("модель")
display(syn_metrics.round(4))

In [ ]:
# --- веса лучшей полиномиальной модели + проверка аналитическим МНК
best_lin = models_syn[f"LinReg, степень {best_deg} (лучшая по val)"]
coef = best_lin.named_steps["model"].coef_
print(f"Лучшая модель: полином степени {best_deg}")
print(f"intercept_ = {best_lin.named_steps['model'].intercept_:.4f}")
print("coef_ (при стандартизованных x^k):")
print(pd.Series(coef, index=[f"x^{k}" for k in range(1, best_deg + 1)]).round(4).to_string())

# аналитическое решение МНК:  w = argmin ||A w - y||^2  (A = [1, X_prepared])
Z = best_lin[:-1].transform(X_tr)
A = np.c_[np.ones(len(Z)), Z]
w_ls = np.linalg.lstsq(A, y_tr, rcond=None)[0]
print(f"\nmax|w_sklearn - w_lstsq| = {np.max(np.abs(np.r_[best_lin.named_steps['model'].intercept_, coef] - w_ls)):.2e}  (≈0 → совпадает)")
n_zero = int(np.sum(np.abs(models_syn[list(models_syn)[-1]].named_steps['model'].coef_) < 1e-12))
print(f"Lasso занулил {n_zero} из {REG_DEG} весов")

In [ ]:
# --- итоговая визуализация модели и остатки на валидации
pred_val = best_lin.predict(X_val)
resid = y_val - pred_val

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
ax = axes[0]
ax.scatter(X_tr, y_tr, s=10, alpha=.35, color="gray", label="train")
ax.scatter(X_val, y_val, s=14, color="tab:orange", label="val")
ax.plot(x_grid, cfg["f"](x_grid), "k-", lw=2, label="истина")
ax.plot(x_grid, best_lin.predict(x_grid), color="tab:green", lw=2, label=f"модель (степень {best_deg})")
ax.set(title="Лучшая модель на фоне данных", xlabel="x", ylabel="y"); ax.legend()

ax = axes[1]
PredictionErrorDisplay.from_predictions(y_val, pred_val, kind="actual_vs_predicted", ax=ax, scatter_kwargs={"alpha": .7, "s": 14})
ax.set_title("Факт vs прогноз (val)")

ax = axes[2]
ax.hist(resid, bins=20, color="tab:green", alpha=.8, edgecolor="white")
ax.axvline(0, color="k", lw=1)
ax.set(title=f"Остатки на val (среднее={resid.mean():.3f}, σ={resid.std():.3f})", xlabel="y − ŷ", ylabel="число точек")
plt.show()

*Комментарий.* Остатки центрированы около нуля и по масштабу сопоставимы с уровнем шума $\sigma$, закономерной структуры в остатках не видно — модель улавливает форму зависимости. Совпадение `coef_` со решением `lstsq` подтверждает, что `LinearRegression` — это именно МНК.

## 3.4 Выводы по синтетическим данным

In [ ]:
r = syn_metrics
lin1 = r.iloc[0]; lin_best = r.iloc[1]; lin15 = r.iloc[2]; rid = r.iloc[3]; las = r.iloc[4]
print(f'''1. Линейной зависимости НЕ хватило: прямая (степень 1) даёт val RMSE = {lin1['val RMSE']:.3f} и R² = {lin1['val R²']:.2f}.
2. Подходящая сложность — полином степени {best_deg}: val RMSE = {lin_best['val RMSE']:.3f} при уровне шума σ = {cfg['noise']}, R² = {lin_best['val R²']:.3f}.
3. Слишком сложная модель (степень {REG_DEG}, без регуляризации) переобучается: train RMSE = {lin15['train RMSE']:.3f}, val RMSE = {lin15['val RMSE']:.3f}.
4. Регуляризация на той же степени {REG_DEG}: без неё val RMSE = {lin15['val RMSE']:.3f}, с Ridge = {rid['val RMSE']:.3f}, с Lasso = {las['val RMSE']:.3f} (Lasso занулил {n_zero} из {REG_DEG} весов).
5. Сложности модели ХВАТИЛО при степени ≈ {best_deg}; дальше рост сложности только увеличивает разрыв train/val.''')

**Итог по части I.** Качество определяется балансом «смещение–дисперсия»: степень полинома и сила регуляризации — две ручки одного и того же регулятора сложности. Регуляризация позволяет брать «с запасом» сложные признаки, не платя переобучением. На функциях с изломом (и у краёв отрезка у функции Рунге) полиномиальный базис принципиально ограничен — там лучше подошли бы сплайны или деревья.

# Часть II. Реальные данные: California Housing (4.1–4.5)

## 4.1 Выбор набора данных

**Датасет:** `California Housing` (перепись США 1990 г., `sklearn.datasets.fetch_california_housing`).

- **Задача регрессии:** целевая переменная `MedHouseVal` — медианная стоимость жилья в блоке, в **сотнях тысяч долларов**; значения непрерывные.
- **Размер:** 20 640 строк (требовалось ≥ 1000) и 8 числовых признаков (требовалось ≥ 5).
- Данные из открытого источника, воспроизводятся одной строкой кода (без ручного скачивания).

| Признак | Смысл |
|---|---|
| `MedInc` | медианный доход жителей блока (десятки тыс. $) |
| `HouseAge` | медианный возраст домов |
| `AveRooms` | среднее число комнат на домохозяйство |
| `AveBedrms` | среднее число спален на домохозяйство |
| `Population` | население блока |
| `AveOccup` | среднее число жителей на домохозяйство |
| `Latitude`, `Longitude` | координаты блока |

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
TARGET = "MedHouseVal"
BASE_NUM = [c for c in df.columns if c != TARGET]

print("Размер:", df.shape)
display(df.head())
display(df.describe().T.round(3))
print("Пропуски:", int(df.isna().sum().sum()), "| Полных дубликатов:", int(df.duplicated().sum()))

### Разведочный анализ (EDA)

Смотрим только то, что влияет на решения по предобработке: распределение таргета, форму признаков и связи.

In [ ]:
CAP = df[TARGET].max()
n_cap = int((df[TARGET] >= CAP - 1e-6).sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df[TARGET], bins=60, color="tab:blue", alpha=.85, edgecolor="white")
axes[0].set(title=f"Таргет: {n_cap} блоков ({n_cap/len(df):.1%}) «обрезаны» на уровне {CAP:.2f}",
            xlabel="MedHouseVal (×100 000 $)", ylabel="число блоков")
axes[1].hist(np.log1p(df[TARGET]), bins=60, color="tab:green", alpha=.85, edgecolor="white")
axes[1].set(title="log(1 + таргет): распределение стало ближе к симметричному", xlabel="log1p(MedHouseVal)")
plt.show()

*Комментарий.* Таргет **скошен вправо** и **цензурирован сверху** (в исходных данных верхняя граница стоимости обрезана) — это важное ограничение: модель не может ни точно предсказать, ни точно оценить дорогие блоки. Логарифм таргета делает распределение симметричнее — это мотивация использовать `TransformedTargetRegressor`.

In [ ]:
skew = df[BASE_NUM].skew().sort_values(ascending=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), skew.index):
    lo, hi = df[col].quantile([0.001, 0.999])        # для читаемости рисуем 0.1–99.9 перцентили
    ax.hist(df[col].clip(lo, hi), bins=50, color="tab:purple", alpha=.8, edgecolor="white")
    ax.set_title(f"{col} (skew={skew[col]:.1f}, max={df[col].max():.0f})" if df[col].max() > 100 else f"{col} (skew={skew[col]:.1f})")
plt.suptitle("Распределения признаков (хвосты обрезаны на 0.1/99.9 перцентиле для рисунка)", y=1.01)
plt.tight_layout(); plt.show()

*Комментарий.* У `AveRooms`, `AveBedrms`, `AveOccup`, `Population` — очень тяжёлые правые хвосты (максимумы на порядки выше типичных значений — вероятно, нежилые/курортные блоки или ошибки учёта). Для линейной модели это опасно: единичные выбросы сильно тянут веса. В предобработке: (1) клиппинг по квантилям, (2) степенное преобразование `PowerTransformer`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.1]})
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            cbar=False, ax=axes[0], annot_kws={"size": 8})
axes[0].set_title("Корреляции (Пирсон)")

sc = axes[1].scatter(df["Longitude"], df["Latitude"], c=df[TARGET], s=3, cmap="viridis", alpha=.6)
plt.colorbar(sc, ax=axes[1], label="MedHouseVal")
axes[1].set(title="Карта: цена жилья (побережье и крупные города дороже)", xlabel="Longitude", ylabel="Latitude")
plt.tight_layout(); plt.show()

*Комментарий.* Сильнее всего с ценой связан доход `MedInc`. Координаты по отдельности почти не коррелируют с ценой линейно, но карта показывает чёткую **географическую структуру** (побережье, Bay Area, Лос-Анджелес) — её линейной модели по сырым `Latitude/Longitude` не уловить. Это идея для новых признаков: расстояние до крупных городов и гео-кластеры.

## 4.2 Предобработка данных

**Разбиение.** 80 % train / 20 % test, фиксируем `random_state`. Тест не трогаем до финальной оценки. Для выбора моделей используем 5-fold CV на train.

**Что делаем (всё внутри `Pipeline`, чтобы не было утечки):**

| Шаг | Зачем |
|---|---|
| `FeatureEngineer` (свой класс) | новые числовые признаки: `RoomsPerPerson`, `BedroomRatio`, `DistNearestCity` (расстояние до ближайшего из 5 крупных городов), повёрнутые на 45° координаты; **категориальные**: `GeoCluster` (KMeans по координатам, обучается на train) и `AgeGroup` (возрастная группа дома) |
| `QuantileClipper` (свой класс) | клиппинг выбросов по квантилям 0.5 % / 99.5 % (границы считаются на train); применяется только к скошенным признакам из EDA |
| `PowerTransformer` | выравнивание скошенных распределений (Йео–Джонсон); координаты и возраст дома не трогаем, они лишь стандартизуются |
| `PolynomialFeatures` | нелинейность (на этапе улучшения) |
| `StandardScaler` | для регуляризации важно единое масштабирование |
| `OneHotEncoder` | кодирование категориальных признаков |
| `ColumnTransformer` | числовые и категориальные признаки обрабатываются по-разному |

In [ ]:
X = df[BASE_NUM]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train: {X_train.shape}, test: {X_test.shape}")

In [ ]:
CITIES = {"SanFrancisco": (37.77, -122.42), "LosAngeles": (34.05, -118.24), "SanDiego": (32.72, -117.16),
          "SanJose": (37.34, -121.89), "Sacramento": (38.58, -121.49)}

def haversine_km(lat1, lon1, lat2, lon2):
    # расстояние по большому кругу в км
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

ENG_NUM = ["RoomsPerPerson", "BedroomRatio", "DistNearestCity", "Rot45_X", "Rot45_Y"]
# признаки с тяжёлыми правыми хвостами (по результатам EDA) — только к ним применяем клиппинг + PowerTransformer
SKEWED = ["MedInc", "AveRooms", "AveBedrms", "Population", "AveOccup", "RoomsPerPerson", "BedroomRatio", "DistNearestCity"]
CAT = ["GeoCluster", "AgeGroup"]

class FeatureEngineer(TransformerMixin, BaseEstimator):
    # Добавляет новые признаки. KMeans обучается ТОЛЬКО на тех данных, что пришли в fit (train / train-fold).
    def __init__(self, n_clusters=20, random_state=SEED):
        self.n_clusters = n_clusters
        self.random_state = random_state

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, n_init=10, random_state=self.random_state)
        self.kmeans_.fit(X[["Latitude", "Longitude"]])
        return self

    def transform(self, X):
        X = X.copy()
        X["RoomsPerPerson"] = X["AveRooms"] / X["AveOccup"]
        X["BedroomRatio"] = X["AveBedrms"] / X["AveRooms"]
        dists = np.column_stack([haversine_km(X["Latitude"], X["Longitude"], la, lo) for la, lo in CITIES.values()])
        X["DistNearestCity"] = dists.min(axis=1)
        X["Rot45_X"] = (X["Latitude"] + X["Longitude"]) / np.sqrt(2)
        X["Rot45_Y"] = (X["Latitude"] - X["Longitude"]) / np.sqrt(2)
        X["GeoCluster"] = self.kmeans_.predict(X[["Latitude", "Longitude"]]).astype(str)
        X["AgeGroup"] = pd.cut(X["HouseAge"], [0, 10, 25, 40, np.inf], labels=["new", "young", "mid", "old"]).astype(str)
        return X


class QuantileClipper(TransformerMixin, BaseEstimator):
    # Обрезает значения по квантилям, посчитанным на train (защита от выбросов).
    def __init__(self, q_low=0.005, q_high=0.995):
        self.q_low = q_low
        self.q_high = q_high

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.cols_ = X.columns.to_numpy()
        self.low_ = X.quantile(self.q_low)
        self.high_ = X.quantile(self.q_high)
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.clip(lower=self.low_, upper=self.high_, axis=1)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.cols_, dtype=object)

In [ ]:
def build_model(regressor, *, clip_power=True, fe=False, cat=False, degree=1):
    # Собирает полный пайплайн. Флаги включают/выключают отдельные улучшения — удобно для абляции.
    num_cols = BASE_NUM + (ENG_NUM if fe else [])
    num_steps = []
    if clip_power:
        skew_cols = [c for c in num_cols if c in SKEWED]
        skew_block = ColumnTransformer(
            [("skew", Pipeline([("clip", QuantileClipper()), ("power", PowerTransformer())]), skew_cols)],
            remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
        num_steps.append(("skew_fix", skew_block))
    num_steps += [("poly", PolynomialFeatures(degree, include_bias=False)), ("scale", StandardScaler())]

    transformers = [("num", Pipeline(num_steps), num_cols)]
    if cat:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT))

    steps = []
    if fe or cat:
        steps.append(("fe", FeatureEngineer()))
    steps += [("prep", ColumnTransformer(transformers)), ("model", regressor)]
    return Pipeline(steps)

# быстрая проверка: пайплайн обучается и отдаёт прогноз нужной формы
_check = build_model(Ridge(), fe=True, cat=True, degree=2).fit(X_train, y_train)
print("Число признаков после предобработки (degree=2, fe+cat):", _check.named_steps["prep"].get_feature_names_out().shape[0])

## 4.3 Baseline: «модель без всего»

Обычная `LinearRegression` на **8 сырых числовых признаках**, без масштабирования, без обработки выбросов, без преобразования таргета.

In [ ]:
results = []     # таблица результатов всех экспериментов
fitted = {}      # обученные модели (на всём train)

def evaluate(name, model):
    # CV на train (для выбора модели) + обучение на всём train + оценка на test (для отчёта)
    cvr = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True,
                         scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"})
    model.fit(X_train, y_train)
    t = metrics(y_test, model.predict(X_test))
    results.append({"Модель": name,
                    "CV RMSE": -cvr["test_rmse"].mean(), "CV RMSE std": cvr["test_rmse"].std(),
                    "Train RMSE": -cvr["train_rmse"].mean(), "CV R²": cvr["test_r2"].mean(),
                    "Test RMSE": t["RMSE"], "Test MAE": t["MAE"], "Test R²": t["R2"]})
    fitted[name] = model
    return model

M0 = "M0 Baseline: LinearRegression, 8 сырых признаков"
baseline = evaluate(M0, LinearRegression())
display(pd.DataFrame(results).set_index("Модель").round(4))

### Артефакты baseline-модели

Веса, метрики и визуализация ошибок. Чтобы веса можно было сравнивать между собой, дополнительно показываем коэффициенты на **стандартизованных** признаках («на сколько σ-таргета меняется прогноз при росте признака на 1σ»).

In [ ]:
coef_raw = pd.DataFrame({"признак": BASE_NUM, "вес (сырые единицы)": baseline.coef_})
print(f"intercept_ = {baseline.intercept_:.4f}")
display(coef_raw.round(4).set_index("признак").T)

# те же веса на стандартизованных признаках — для честного сравнения «важности»
lr_std = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]).fit(X_train, y_train)
coef_std = pd.Series(lr_std.named_steps["model"].coef_, index=BASE_NUM).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
colors = ["tab:red" if v < 0 else "tab:blue" for v in coef_std.values]
axes[0].barh(coef_std.index, coef_std.values, color=colors)
axes[0].set(title="Baseline: веса на стандартизованных признаках", xlabel="вес")

PredictionErrorDisplay.from_estimator(baseline, X_test, y_test, kind="actual_vs_predicted", ax=axes[1],
                                      scatter_kwargs={"alpha": .25, "s": 6})
axes[1].set_title("Baseline: факт vs прогноз (test)")
PredictionErrorDisplay.from_estimator(baseline, X_test, y_test, kind="residual_vs_predicted", ax=axes[2],
                                      scatter_kwargs={"alpha": .25, "s": 6})
axes[2].set_title("Baseline: остатки vs прогноз (test)")
plt.tight_layout(); plt.show()

*Комментарий.* По стандартизованным весам видно, какие признаки сильнее всего двигают прогноз (ожидаемо — `MedInc`; координаты участвуют в модели только линейно и потому плохо описывают реальную карту цен). На графике «факт vs прогноз» обращаем внимание на **горизонтальную «полку»** из блоков с обрезанной ценой и на то, как прогноз ведёт себя на дорогом жилье; на графике остатков любая кривизна или «веер» означает, что линейная модель не улавливает нелинейность или что дисперсия ошибок растёт с ценой. Это направления для улучшения в 4.4.

## 4.4 Улучшение качества модели

Идём по «лесенке»: каждый шаг добавляет **одну** идею, поэтому по таблице видно вклад каждой (абляция).

| Шаг | Что добавляем |
|---|---|
| M1 | клиппинг выбросов + `PowerTransformer` для скошенных признаков |
| M2 | + новые числовые признаки (расстояния до городов, отношения, повороты координат) |
| M3 | + категориальные признаки (`GeoCluster`, `AgeGroup`) с one-hot |
| M4 | + полиномы степени 2 и регуляризация Ridge |
| M5 | + логарифм таргета через `TransformedTargetRegressor` |

In [ ]:
M1 = "M1 + clip и PowerTransformer (скошенные признаки)"
M2 = "M2 + новые числовые признаки"
M3 = "M3 + категориальные (GeoCluster, AgeGroup)"
M4 = "M4 + полиномы deg=2, Ridge(α=10)"
M5 = "M5 + log-таргет (TransformedTargetRegressor)"

evaluate(M1, build_model(LinearRegression()))
evaluate(M2, build_model(LinearRegression(), fe=True))
evaluate(M3, build_model(LinearRegression(), fe=True, cat=True))
evaluate(M4, build_model(Ridge(alpha=10), fe=True, cat=True, degree=2))
evaluate(M5, TransformedTargetRegressor(regressor=build_model(Ridge(alpha=10), fe=True, cat=True, degree=2),
                                        func=np.log1p, inverse_func=np.expm1))
display(pd.DataFrame(results).set_index("Модель").round(4))

### Подбор гиперпараметров: степень полинома и сила регуляризации

`GridSearchCV` по 5-fold CV на train: степень ∈ {1, 2, 3}, α ∈ {0.1, 1, 10, 100, 1000}. Логарифм таргета включается в тюнинг **только если он улучшил CV на шаге M5** — решение принимается автоматически по данным.

In [ ]:
# логарифм таргета берём в тюнинг только если он улучшил CV на шаге M5 (решение автоматическое)
use_log = results[-1]["CV RMSE"] < results[-2]["CV RMSE"]
P = "regressor__" if use_log else ""          # префикс имён параметров внутри TransformedTargetRegressor
def maybe_log(pipe):
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1) if use_log else pipe
def inner(model):
    return model.regressor_ if isinstance(model, TransformedTargetRegressor) else model
print("Логарифм таргета в тюнинге:", "ДА (улучшил CV на M5)" if use_log else "НЕТ (на M5 не улучшил CV)")

grid = GridSearchCV(maybe_log(build_model(Ridge(), fe=True, cat=True, degree=2)),
                    param_grid={P + "prep__num__poly__degree": [1, 2, 3],
                                P + "model__alpha": [0.1, 1, 10, 100, 1000]},
                    scoring="neg_root_mean_squared_error", cv=cv)
grid.fit(X_train, y_train)

gres = pd.DataFrame(grid.cv_results_)
gres["степень"] = gres["param_" + P + "prep__num__poly__degree"].astype(int)
gres["alpha"] = gres["param_" + P + "model__alpha"].astype(float)
gres["CV RMSE"] = -gres["mean_test_score"]
pivot = gres.pivot(index="степень", columns="alpha", values="CV RMSE")

fig, ax = plt.subplots(figsize=(7, 3.2))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="viridis_r", ax=ax, cbar_kws={"label": "CV RMSE"})
ax.set_title("GridSearchCV: CV RMSE (меньше — лучше)")
plt.show()

best_degree = int(grid.best_params_[P + "prep__num__poly__degree"])
best_alpha = float(grid.best_params_[P + "model__alpha"])
print("Лучшие параметры:", grid.best_params_, f"| CV RMSE = {-grid.best_score_:.4f}")

### Другие модели и регуляризация: Lasso и SGDRegressor

- **Lasso** (L1) на полиномах — заодно отбирает признаки, считаем, сколько весов обнулилось;
- **SGDRegressor** — та же линейная регрессия, но обучаемая градиентным спуском (как в демонстрационном блокноте).

Для скорости полиномы берём степени ≤ 2.

In [ ]:
M6 = f"M6 Ridge: подобранные deg={best_degree}, α={best_alpha:g} (GridSearch{', log-таргет' if use_log else ''})"
evaluate(M6, maybe_log(build_model(Ridge(alpha=best_alpha), fe=True, cat=True, degree=best_degree)))

deg_l = min(best_degree, 2)
lasso_grid = GridSearchCV(
    maybe_log(build_model(Lasso(max_iter=20000, tol=1e-3), fe=True, cat=True, degree=deg_l)),
    param_grid={P + "model__alpha": [1e-4, 1e-3, 1e-2]},
    scoring="neg_root_mean_squared_error", cv=cv).fit(X_train, y_train)
a_l = float(lasso_grid.best_params_[P + "model__alpha"])
M7 = f"M7 Lasso: deg={deg_l}, α={a_l:g}"
evaluate(M7, maybe_log(build_model(Lasso(alpha=a_l, max_iter=20000, tol=1e-3), fe=True, cat=True, degree=deg_l)))
lasso_coefs = inner(fitted[M7]).named_steps["model"].coef_
print(f"Lasso: обнулено {int((lasso_coefs == 0).sum())} из {lasso_coefs.size} весов")

M8 = f"M8 SGDRegressor (L2): deg={deg_l}"
evaluate(M8, maybe_log(build_model(SGDRegressor(penalty="l2", alpha=1e-4, max_iter=3000, tol=1e-4, random_state=SEED),
                                   fe=True, cat=True, degree=deg_l)))

res_df = pd.DataFrame(results).set_index("Модель")
display(res_df.round(4))
res_df.round(5).to_csv("results_summary.csv")

In [ ]:
# сводный график: CV RMSE (по которому выбираем) и Test RMSE (для отчёта)
plot_df = res_df[["CV RMSE", "Test RMSE"]]
fig, ax = plt.subplots(figsize=(11, 4.8))
plot_df.plot.barh(ax=ax, width=.75, color=["tab:blue", "tab:orange"])
ax.invert_yaxis()
ax.set(title="Сравнение моделей по RMSE (×100 000 $)", xlabel="RMSE", ylabel="")
lo = plot_df.min().min(); hi = plot_df.max().max()
ax.set_xlim(max(0, lo - 0.05), hi + 0.03)
plt.tight_layout(); plt.show()

*Комментарий.* Таблица показывает вклад каждой идеи. Для выбора лучшей модели используется **CV RMSE** (колонка, которую модель не «видела» при обучении фолдов), а Test RMSE — независимая контрольная оценка. Близость CV и Test говорит об отсутствии «подгонки под тест». Разрыв между `Train RMSE` и `CV RMSE` показывает, нет ли переобучения при росте степени полинома.

## 4.5 Финальная модель, анализ ошибок и выводы

Финальная модель выбирается **автоматически** по минимальному CV RMSE, затем один раз оценивается на тесте.

In [ ]:
final_name = res_df["CV RMSE"].idxmin()
final = fitted[final_name]
base = fitted[M0]
print("Финальная модель:", final_name)

pred_b = base.predict(X_test)
pred_f = final.predict(X_test)
display(pd.DataFrame({"Baseline": metrics(y_test, pred_b), "Финал": metrics(y_test, pred_f)}).T.round(4))

# --- артефакты финальной модели: самые «тяжёлые» веса
est = inner(final)
names = est.named_steps["prep"].get_feature_names_out()
w = pd.Series(est.named_steps["model"].coef_, index=names)
print(f"intercept_ = {float(np.ravel(est.named_steps['model'].intercept_)[0]):.4f};  всего весов: {len(w)}")
top = w.reindex(w.abs().sort_values(ascending=False).index[:15]).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([n.replace("num__", "").replace("cat__", "") for n in top.index], top.values,
        color=["tab:red" if v < 0 else "tab:blue" for v in top.values])
ax.set(title="Финальная модель: 15 самых больших по модулю весов\n(признаки стандартизованы" + ("; вес относится к log1p-таргету)" if isinstance(final, TransformedTargetRegressor) else ")"), xlabel="вес")
plt.tight_layout(); plt.show()

In [ ]:
# --- сравнение ошибок baseline и финальной модели
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row, (nm, mdl) in enumerate([("Baseline", base), ("Финал", final)]):
    PredictionErrorDisplay.from_estimator(mdl, X_test, y_test, kind="actual_vs_predicted", ax=axes[row, 0],
                                          scatter_kwargs={"alpha": .25, "s": 6})
    axes[row, 0].set_title(f"{nm}: факт vs прогноз")
    PredictionErrorDisplay.from_estimator(mdl, X_test, y_test, kind="residual_vs_predicted", ax=axes[row, 1],
                                          scatter_kwargs={"alpha": .25, "s": 6})
    axes[row, 1].set_title(f"{nm}: остатки vs прогноз")
plt.tight_layout(); plt.show()

*Комментарий.* Сравнение «до/после»: смотрим, стало ли облако остатков ближе к горизонтальной полосе вокруг нуля. «Полка» из блоков с обрезанной ценой по построению остаётся в любой модели — это свойство данных (цензурирование таргета), а не недообучение.

In [ ]:
# --- где модель ошибается: по сегментам цены и на карте
res_test = pd.DataFrame({"y": y_test.values, "abs_err_base": np.abs(y_test.values - pred_b),
                         "abs_err_final": np.abs(y_test.values - pred_f), "resid_final": y_test.values - pred_f,
                         "lat": X_test["Latitude"].values, "lon": X_test["Longitude"].values})
res_test["сегмент цены"] = pd.cut(res_test["y"], [0, 1, 2, 3, 4, np.inf],
                                   labels=["<100k", "100–200k", "200–300k", "300–400k", "≥400k (вкл. обрезанные)"])
seg = res_test.groupby("сегмент цены", observed=True)[["abs_err_base", "abs_err_final"]].mean()
seg.columns = ["Baseline", "Финал"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
seg.plot.bar(ax=axes[0], color=["tab:gray", "tab:green"], rot=20)
axes[0].set(title="MAE по сегментам реальной цены", ylabel="MAE (×100 000 $)", xlabel="")

axes[1].hist(y_test.values - pred_b, bins=60, alpha=.55, label="Baseline", color="tab:gray")
axes[1].hist(res_test["resid_final"], bins=60, alpha=.6, label="Финал", color="tab:green")
axes[1].axvline(0, color="k", lw=1)
axes[1].set(title="Распределение остатков (test)", xlabel="y − ŷ", ylabel="число блоков"); axes[1].legend()

sc = axes[2].scatter(res_test["lon"], res_test["lat"], c=res_test["resid_final"], s=6, cmap="coolwarm", vmin=-1.5, vmax=1.5)
plt.colorbar(sc, ax=axes[2], label="остаток y − ŷ")
axes[2].set(title="Карта остатков финальной модели\n(синий — недооценка, красный — переоценка)", xlabel="Longitude", ylabel="Latitude")
plt.tight_layout(); plt.show()

*Комментарий.* Левый график показывает, **одинакова ли ошибка по диапазону цен** (рост MAE на краях — признак гетероскедастичности и цензурирования таргета). Карта остатков показывает, остались ли географические области систематической ошибки (синие/красные пятна) — это признак того, что линейная модель не до конца описала локальные ценовые эффекты.

In [ ]:
# --- сохранение / загрузка модели (как в демонстрационном блокноте)
joblib.dump(final, "california_housing_model.joblib")
loaded = joblib.load("california_housing_model.joblib")
print("Загруженная модель даёт те же прогнозы:", bool(np.allclose(loaded.predict(X_test), pred_f)))

### Выводы

In [ ]:
b = res_df.loc[M0]; f = res_df.loc[final_name]
gain = (b["Test RMSE"] - f["Test RMSE"]) / b["Test RMSE"]
within = lambda p: float(np.mean(np.abs(y_test.values - p) <= 0.5))
worst_seg = seg["Финал"].idxmax()
print(f'''1. МЕТРИКИ. Baseline: Test RMSE = {b["Test RMSE"]:.3f} (≈ {b["Test RMSE"]*100:.0f} тыс. $), MAE = {b["Test MAE"]:.3f}, R² = {b["Test R²"]:.3f}.
   Финал ({final_name}): Test RMSE = {f["Test RMSE"]:.3f} (≈ {f["Test RMSE"]*100:.0f} тыс. $), MAE = {f["Test MAE"]:.3f}, R² = {f["Test R²"]:.3f}.
   RMSE снизился на {gain:.1%}; R² вырос на {f["Test R²"] - b["Test R²"]:+.3f}.
2. АДЕКВАТНОСТЬ. Модель объясняет {f["Test R²"]:.0%} дисперсии цены; в пределах ±50 тыс. $ оказывается {within(pred_f):.0%} блоков (у baseline — {within(pred_b):.0%}).
3. ПЕРЕОБУЧЕНИЕ. Train RMSE (CV) = {f["Train RMSE"]:.3f} против CV RMSE = {f["CV RMSE"]:.3f} и Test RMSE = {f["Test RMSE"]:.3f}: разрыв train/CV = {f["CV RMSE"] - f["Train RMSE"]:.3f}, CV и Test согласованы → {"сильного переобучения нет" if (f["CV RMSE"] - f["Train RMSE"]) < 0.05 else "есть заметное переобучение, стоит усилить регуляризацию"}.
4. ОШИБКИ. Хуже всего модель предсказывает сегмент «{worst_seg}» (MAE = {seg["Финал"].max():.3f}); лучше всего — «{seg["Финал"].idxmin()}» (MAE = {seg["Финал"].min():.3f}).''')
print("5. АБЛЯЦИЯ (изменение CV RMSE при добавлении очередной идеи; меньше нуля — лучше):")
_steps = [M0, M1, M2, M3, M4, M5]
for _a, _b in zip(_steps[:-1], _steps[1:]):
    _d = res_df.loc[_b, "CV RMSE"] - res_df.loc[_a, "CV RMSE"]
    print(f"   {_b}: ΔCV RMSE = {_d:+.4f} ({'лучше' if _d < 0 else 'хуже'})")

**Содержательные выводы**

- **Что дало прирост.** Вклад каждой идеи (выбросы + степенное преобразование, гео-признаки, категориальные признаки, полиномы, лог-таргет) виден в таблице абляции и в автоматическом выводе выше (ΔCV RMSE). Не каждая идея обязана улучшать метрику: например, логарифм таргета улучшает симметрию остатков, но RMSE в исходной шкале может не снизиться — модель начинает предсказывать скорее медиану, чем среднее; поэтому финальная модель выбирается по CV, а не «по ощущению».
- **Регуляризация.** Ridge/Lasso позволяют брать полиномиальные признаки без катастрофического переобучения; GridSearch показывает, что слишком большая степень или слишком слабая регуляризация ухудшают CV-качество.
- **Адекватность метрик.** RMSE и MAE интерпретируются в долларах (×100 000), R² показывает долю объяснённой дисперсии. Метрики адекватные, но их нужно читать вместе с **цензурированием таргета**: часть самых дорогих блоков обрезана на верхней границе, поэтому ошибки в этом сегменте занижают реальную проблему.
- **Распределение ошибок.** Остатки не идеально нормальны и не равномерны по цене и по карте (гетероскедастичность, региональные кластеры) → допущения классической линейной регрессии выполняются лишь приближённо.
- **Ограничения и что дальше.** Линейная модель не умеет в сложные взаимодействия «координаты × доход»; следующим шагом логично попробовать деревья/градиентный бустинг, сплайны по координатам или модели с гео-эмбеддингами, а также обработать обрезанные значения таргета (например, цензурированной регрессией).